In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  
proxy = 'http://10.20.38.38:7890'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
import sys
import json
import torch
sys.path.append('/home/ldy/Closed_loop_optimizing/experiments')
sys.path.append('/home/ldy/Closed_loop_optimizing/model')
from sklearn.decomposition import PCA
from ATMS_retrieval import get_eeg_features, ATMS
from util import (plot_similarity_and_mse_with_dual_axis, get_gteeg, save_eeg, load_thingstestimagedata, get_image_path)
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# image_folder = '/mnt/dataset0/kyw/closed-loop/image_select'
image_folder = '/mnt/dataset0/jiahua/closed-loop/imageset'
# '/mnt/dataset0/kyw/closed-loop/image_select'#'/mnt/dataset0/ldy/NSD_data/pictures/shared1000'
text_list = sorted(os.listdir(image_folder))
syn_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/results/syn_eeg'
# 过滤掉隐藏文件或其他不需要的文件（如果有）
text_list = [file for file in text_list if not file.startswith('.')]
# text_list

In [ ]:

image_gt_folder = ['/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00014_bike/bike_14s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00181_television/television_14n.jpg'
                        , '/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00177_t-shirt/t-shirt_13s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00135_pie/pie_15s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00131_pear/pear_13s.jpg']


image_gt_path = image_gt_folder[1]

dir_name = os.path.basename(os.path.dirname(image_gt_path))  # '00014_bike'
gt_category_id = text_list.index(dir_name) 
gt_category_id

In [ ]:
dnn = 'alexnet' #'alexnet' #,'cornet_s'
sub = 'sub-01'
# encoder_model_path = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
encoder_model_path = f'/mnt/dataset0/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
gt_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/syn_eeg_gt'
# gt_eeg_folder = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/data/syn_eeg_gt'

In [ ]:
seed = 4  # 你可以根据需要更改随机种子

max_iterations = 50
gamma = 0.9
num_images= 10

os.makedirs(gt_eeg_folder, exist_ok=True)
gt_eeg_path = os.path.join(gt_eeg_folder, f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy")
print(f"gt_eeg_path {gt_eeg_path}")
try:
    synthetic_eeg = torch.from_numpy(np.load(gt_eeg_path))
except:
    synthetic_eeg = torch.tensor(get_gteeg(image_gt_path, encoder_model_path, dnn, device)) 
gt_eeg_path = save_eeg(synthetic_eeg, gt_eeg_folder, file_name=f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy" )    

synthetic_eeg.shape


# f_encoder =  f"/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/sub_model/{sub}/diffusion_alexnet/pretrained_True/gene_gene/ATM_S_reconstruction_scale_0_1000_40.pth"
f_encoder =  f"/mnt/dataset0/kyw/closed-loop/sub_model/{sub}/diffusion_alexnet/pretrained_True/gene_gene/ATM_S_reconstruction_scale_0_1000_40.pth"
checkpoint = torch.load(f_encoder, map_location=device)

eeg_model = ATMS()  
eeg_model.load_state_dict(checkpoint['eeg_model_state_dict'])
yhat = get_eeg_features(eeg_model, synthetic_eeg.unsqueeze(0), device, sub)


In [ ]:
def load_similarities_json(save_folder, filename="similarities.json"):
    filepath = os.path.join(save_folder, filename)
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"File {filepath} not found")
        return None


In [ ]:
def visualize_feature_distribution(yhat, selected_eegs_per_iteration, save_folder):
    """
    可视化EEG特征分布，随着迭代展示10个EEG相对于目标的位置，颜色随着迭代变深，且标记每次迭代最接近目标的点。
    :param yhat: 目标EEG特征向量，形状为 [1, 1024]
    :param selected_eegs_per_iteration: 每次迭代选择的EEG特征，形状为 [iterations, 10, 1, 1024]
    :param save_folder: 图像保存的文件夹
    """

    # 将目标 EEG 特征从 [1, 1024] 转换为 [1024]，并移除梯度追踪
    yhat = yhat.squeeze().detach()
    # plt.xlabel('t-SNE dimension 2', fontsize=14)
    # plt.ylabel('t-SNE dimension 1', fontsize=14)

    # 将目标 EEG 特征和每轮选中的 EEG 特征合并，用于 PCA 降维
    all_eegs = np.vstack([yhat.cpu().numpy()] + 
                         [np.array([feature.squeeze().detach().cpu().numpy() for feature in features]) 
                          for features in selected_eegs_per_iteration])

    # 使用 PCA 将 EEG 特征降维到 2D
    pca = PCA(n_components=2)
    all_eegs_2d = pca.fit_transform(all_eegs)

    # 获取降维后的目标 EEG 特征
    target_feature_2d = all_eegs_2d[0]

    # 获取每次迭代中选择的 EEG 特征（已降维）
    start_idx = 1
    selected_eegs_2d_per_iteration = []
    for i in range(len(selected_eegs_per_iteration)):
        end_idx = start_idx + len(selected_eegs_per_iteration[i])
        selected_eegs_2d_per_iteration.append(all_eegs_2d[start_idx:end_idx])
        start_idx = end_idx

    plt.figure(figsize=(10, 8))

    # 使用 'viridis' 连续调色板，并反转颜色，颜色跨度更大
    cmap = plt.cm.get_cmap('viridis', len(selected_eegs_per_iteration)).reversed()  # 生成 len(selected_eegs_per_iteration) 个不同颜色
    iterations = len(selected_eegs_per_iteration)

    # 绘制每轮选中的EEG
    for i in range(iterations):
        selected_features_2d = selected_eegs_2d_per_iteration[i]
        
        # 计算每个点到目标的欧几里得距离
        distances = np.linalg.norm(selected_features_2d - target_feature_2d, axis=1)
        
        # 将距离归一化为 0 到 1 之间
        norm_distances = (distances - distances.min()) / (distances.max() - distances.min() + 1e-6)

        # 获取当前迭代的颜色
        base_color = cmap(i / iterations)

        # 找到每轮中距离最近的点
        closest_idx = np.argmin(distances)
        
        # 绘制每轮中9个点（半透明，点越远越小）
        for j in range(len(selected_features_2d)):
            if j != closest_idx:  # 不绘制最近的点
                size = 100 * (1 - norm_distances[j]) + 40  # 大小参数乘以2（原50→100，20→40）
                alpha = 0.3  # 半透明
                plt.scatter(selected_features_2d[j, 0], selected_features_2d[j, 1], color=base_color, s=size, alpha=alpha)
        
        # 绘制每轮中距离最近的点（正常颜色，正常大小）
        plt.scatter(selected_features_2d[closest_idx, 0], selected_features_2d[closest_idx, 1], color=base_color, s=300, alpha=1.0, label=f'Iteration {i+1}')  # 大小从150→300

    # 确保目标点在最上层，且不被覆盖
    plt.scatter(target_feature_2d[0], target_feature_2d[1], color='red', label='Target', s=600, marker='*', zorder=5)  # 大小从300→600

    # 增加网格线
    # plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    # 设置标题和轴标签
    # plt.title('EEG Feature Distribution Over Iterations', fontsize=16)
    plt.xlabel('t-SNE dimenstion2', fontsize=25)
    plt.ylabel('t-SNE dimenstion1', fontsize=25)

    # 调整图例，右侧只显示迭代次数
    plt.legend(loc='upper left', fontsize=25, bbox_to_anchor=(1.05, 1), borderaxespad=0., frameon=False)
    plt.tick_params(axis='both', which='major', labelsize=20)
    # 保存图像
    save_path = os.path.join(save_folder, f"EEG_Feature_Distribution_Over_Iterations.png")
    plt.savefig(save_path,  dpi=300)  # 保存图像文件为高清
    print(f"Visualization saved to {save_path}")
    
    # 显示图形并关闭
    plt.show()
    plt.close()  # 关闭图形，释放内存


In [ ]:
import numpy as np

plots_save_folder = '/home/ldy/Closed_loop_optimizing/plots/Interactive_search'
save_folder = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/outputs'


similarities_per_iteration = load_similarities_json(plots_save_folder, filename="Interactive_search.json") 
selected_eegs_per_iteration = torch.load("/home/ldy/Closed_loop_optimizing/plots/Interactive_search/stacked_eeg_features.pt").cpu()
visualize_feature_distribution(yhat, selected_eegs_per_iteration, plots_save_folder)
# loooop_max_similarities.append(max_similarities)
# save_amx_similarities(save_folder, loooop_max_similarities)